# Qwen3.5-VL COMFORT prompt-ablation result analysis

This notebook analyzes the complete precise-prompt series under `test*/results_preciseprompt/`, summarizes accuracy, visualizes semantic confusion matrices, and analyzes accuracy by camera viewpoint and person-centric object relation parsed from each image filename. `test00_baseline/results/` is used as the shared no-prompt baseline. Alternate-prompt files under `results/` for tests 01–04 are reported as separate variants and never mixed with the precise-prompt series. The quality table explicitly marks partial runs.

## 1. Imports and configuration

Required packages: `pandas`, `numpy`, `matplotlib`, and `seaborn`. Run the notebook from either the repository root or the `comfort_addionalprompt_tests` directory. Set `EXPORT_TABLES=True` to save the summary tables.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
RELATION_ORDER = ["front", "behind", "left", "right"]
CAMERA_ORDER = ["back", "front", "left", "right"]
EXPORT_TABLES = False

ROOT = Path.cwd().resolve()
if ROOT.name == "comfort_addionalprompt_tests":
    ROOT = ROOT.parent
RESULT_ROOT = ROOT / "comfort_addionalprompt_tests"
assert RESULT_ROOT.is_dir(), "Run this notebook from the repository root or comfort_addionalprompt_tests/."

def print_table(table):
    """Print a DataFrame without IPython display or pandas Styler."""
    print(table.to_string(index=False))
print(f"Repository root: {ROOT}")

## 2. Discover and load every Qwen3.5 result

The main series is read from every `results_preciseprompt` directory. The shared test00 baseline and the explicitly listed alternate-prompt tests 01–04 are then added. Files ending in `_thinking.csv` are labeled `thinking`; the others are `non-thinking`.

In [ ]:
def result_record(path, test_name, test_number, prompt_variant, variant_rank):
    return {
        "base_test": test_name,
        "test": test_name if prompt_variant == "baseline" else f"{test_name} [{prompt_variant}]",
        "test_number": test_number,
        "variant_rank": variant_rank,
        "prompt_variant": prompt_variant,
        "mode": "thinking" if path.stem.endswith("_thinking") else "non-thinking",
        "path": path,
    }

records = []

# Main precise-prompt series: test01, test02, ..., including test06 when results exist.
for path in sorted(RESULT_ROOT.glob("test*/results_preciseprompt/mcq_long_qwen3_5vl*.csv")):
    test_name = path.parents[1].name
    test_number = int(re.match(r"test(\d+)", test_name).group(1))
    records.append(result_record(path, test_name, test_number, "precise_prompt", 1))

# Shared no-prompt baseline.
baseline = RESULT_ROOT / "test00_baseline/results/mcq_long_qwen3_5vl.csv"
records.append(result_record(baseline, "test00_baseline", 0, "baseline", 0))

# Alternate prompt formulation. These are intentionally separate from precise_prompt.
ALTERNATE_TESTS = {
    1: "test01_camera_side",
    2: "test02_camera_coordinate_system",
    3: "test03_object_camera_xyz",
    4: "test04_person_object_camera_xyz",
}
for test_number, test_name in ALTERNATE_TESTS.items():
    for path in sorted((RESULT_ROOT / test_name / "results").glob("mcq_long_qwen3_5vl*.csv")):
        records.append(result_record(path, test_name, test_number, "alternate_prompt", 2))

result_files = (pd.DataFrame(records)
                .sort_values(["mode", "test_number", "variant_rank"])
                .reset_index(drop=True))
print_table(result_files.assign(path=result_files["path"].astype(str)))


### Filename-derived viewpoint logic

A path such as `behind/basketball__behind__cam_left/0.png` is interpreted as follows:

- the directory / middle filename token (`behind`) gives the object's **person-centric relation**;
- `cam_left` gives the camera's location relative to the person;
- aliases such as `infrontof`, `front`, `back`, and `rear` are normalized;
- the parser checks whether the directory relation agrees with the filename relation and flags disagreements.

The predicted answer letter is converted back to a semantic direction by reading the randomized MCQ option text. Therefore confusion matrices remain valid even when option letters differ across runs.

In [ ]:
RELATION_ALIASES = {
    "front": "front", "infront": "front", "infrontof": "front",
    "behind": "behind", "back": "behind", "rear": "behind",
    "left": "left", "right": "right",
}
CAMERA_ALIASES = {
    "back": "back", "behind": "back", "rear": "back",
    "front": "front", "infront": "front", "infrontof": "front",
    "left": "left", "right": "right",
}

def normalize_token(value, aliases):
    token = re.sub(r"[^a-z]", "", str(value).lower())
    return aliases.get(token, "unknown")

def parse_image_viewpoint(image_path):
    path = str(image_path).replace("\\", "/")
    parts = [part for part in path.split("/") if part]
    filename_parent = parts[-3] if len(parts) >= 3 else ""
    scene_dir = parts[-2] if len(parts) >= 2 else ""
    tokens = scene_dir.split("__")

    object_name = tokens[0] if tokens else "unknown"
    filename_relation = normalize_token(tokens[1], RELATION_ALIASES) if len(tokens) > 1 else "unknown"
    camera_match = re.search(r"(?:^|__)cam_([a-z]+)(?:__|$)", scene_dir.lower())
    camera_view = normalize_token(camera_match.group(1), CAMERA_ALIASES) if camera_match else "unknown"
    directory_relation = normalize_token(filename_parent, RELATION_ALIASES)
    person_relation = filename_relation if filename_relation != "unknown" else directory_relation

    return pd.Series({
        "parsed_object": object_name,
        "person_relation_from_path": person_relation,
        "camera_view_from_path": camera_view,
        "directory_relation": directory_relation,
        "path_relation_consistent": (directory_relation == filename_relation)
            if "unknown" not in (directory_relation, filename_relation) else pd.NA,
    })

def predicted_relation(row):
    letter = str(row.get("pred_letter", "")).strip().upper()
    if letter not in {"A", "B", "C", "D"}:
        return "invalid"
    option_pattern = re.compile(rf"^{letter}\.\s*(.*)$", re.MULTILINE | re.IGNORECASE)
    match = option_pattern.search(str(row.get("mcq_prompt", "")))
    if not match:
        return "invalid"
    option = match.group(1).lower()
    if "in front" in option:
        return "front"
    if "behind" in option:
        return "behind"
    if "left" in option:
        return "left"
    if "right" in option:
        return "right"
    return "invalid"

frames = []
for meta in result_files.to_dict(orient="records"):
    frame = pd.read_csv(meta["path"])
    frame["test"] = meta["test"]
    frame["base_test"] = meta["base_test"]
    frame["prompt_variant"] = meta["prompt_variant"]
    frame["variant_rank"] = meta["variant_rank"]
    frame["test_number"] = meta["test_number"]
    frame["mode"] = meta["mode"]
    frame["source_file"] = str(meta["path"])
    frame = pd.concat([frame, frame["image_path"].apply(parse_image_viewpoint)], axis=1)
    frame["true_relation"] = frame["correct_relation"].map(lambda x: normalize_token(x, RELATION_ALIASES))
    frame["predicted_relation"] = frame.apply(predicted_relation, axis=1)
    frame["is_correct"] = frame["pred_letter"].fillna("").str.strip().eq(
        frame["correct_letter"].fillna("").str.strip())
    frame["opposite_error"] = frame["pred_letter"].fillna("").str.strip().eq(
        frame["opposite_letter"].fillna("").str.strip())
    frames.append(frame)

results = pd.concat(frames, ignore_index=True)
results["run"] = results["test"] + " | " + results["mode"]
print(f"Loaded {len(results):,} predictions from {len(result_files)} CSV files.")
print_table(results[["test", "mode", "image_path", "person_relation_from_path",
                     "camera_view_from_path", "true_relation", "predicted_relation",
                     "is_correct"]].head())

## 3. Data-quality checks

This section reports missing/invalid predictions, duplicate image rows, filename parsing failures, and disagreement between ground truth stored in the CSV and the relation encoded in the image path.

In [ ]:
quality = (results.groupby(["test_number", "test", "mode"], dropna=False)
    .apply(lambda g: pd.Series({
        "rows": len(g),
        "expected_rows": 144,
        "missing_expected_rows": max(0, 144 - len(g)),
        "unique_images": g["image_path"].nunique(),
        "duplicate_image_rows": g["image_path"].duplicated().sum(),
        "missing_pred_letter": g["pred_letter"].isna().sum() + g["pred_letter"].fillna("").eq("").sum(),
        "invalid_semantic_prediction": g["predicted_relation"].eq("invalid").sum(),
        "unknown_camera_view": g["camera_view_from_path"].eq("unknown").sum(),
        "unknown_person_relation": g["person_relation_from_path"].eq("unknown").sum(),
        "path_vs_csv_relation_mismatch": g["person_relation_from_path"].ne(g["true_relation"]).sum(),
        "inconsistent_path_tokens": g["path_relation_consistent"].eq(False).sum(),
    }))
    .reset_index().sort_values(["mode", "test_number"]))
print_table(quality)

## 4. Overall accuracy summary

Wilson 95% confidence intervals are shown for each result file. `opposite_error_rate` is the fraction of all questions for which the model selected the exact opposite direction.

In [ ]:
def wilson_interval(correct, total, z=1.96):
    if total == 0:
        return np.nan, np.nan
    p = correct / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt(p * (1-p) / total + z**2 / (4 * total**2)) / denominator
    return center - margin, center + margin

summary_rows = []
for (number, test, mode), group in results.groupby(["test_number", "test", "mode"], sort=False):
    n = len(group)
    correct = int(group["is_correct"].sum())
    low, high = wilson_interval(correct, n)
    summary_rows.append({
        "test_number": number, "test": test, "mode": mode,
        "prompt_variant": group["prompt_variant"].iloc[0],
        "correct": correct, "total": n, "accuracy": correct/n,
        "ci95_low": low, "ci95_high": high,
        "opposite_errors": int(group["opposite_error"].sum()),
        "opposite_error_rate": group["opposite_error"].mean(),
    })
summary = pd.DataFrame(summary_rows).sort_values(["mode", "test_number"]).reset_index(drop=True)
summary_display = summary.copy()
for column in ["accuracy", "ci95_low", "ci95_high", "opposite_error_rate"]:
    summary_display[column] = summary_display[column].map(lambda value: f"{value:.2%}")
print_table(summary_display)

fig, ax = plt.subplots(figsize=(max(10, len(summary) * 0.8), 5))
plot_data = summary.copy()
plot_data["label"] = plot_data["test"] + "\n" + plot_data["mode"]
errors = np.vstack([plot_data["accuracy"] - plot_data["ci95_low"],
                    plot_data["ci95_high"] - plot_data["accuracy"]])
colors = plot_data["mode"].map({"non-thinking": "#4C78A8", "thinking": "#F58518"})
ax.bar(plot_data["label"], plot_data["accuracy"], color=colors, yerr=errors, capsize=4)
ax.axhline(0.25, color="black", linestyle="--", linewidth=1, label="4-choice chance = 25%")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, max(0.55, float(plot_data["ci95_high"].max()) + 0.05))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.set_title("Qwen3.5-VL accuracy across prompt-ablation tests")
plt.tight_layout()
plt.show()

## 5. Semantic confusion matrices

Rows are true person-centric relations and columns are predicted semantic relations. The first figure shows counts; the second is row-normalized, so each cell is the percentage of one true direction predicted as each direction. Change `CONFUSION_MODE` to `thinking`, `non-thinking`, or `all`.

In [ ]:
CONFUSION_MODE = "non-thinking"  # "thinking", "non-thinking", or "all"

def confusion_table(group, normalize=False):
    prediction_order = RELATION_ORDER + (["invalid"] if group["predicted_relation"].eq("invalid").any() else [])
    matrix = pd.crosstab(group["true_relation"], group["predicted_relation"])
    matrix = matrix.reindex(index=RELATION_ORDER, columns=prediction_order, fill_value=0)
    if normalize:
        matrix = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0)
    return matrix

def plot_confusion_matrices(data, mode="non-thinking", normalize=False):
    selected = data if mode == "all" else data[data["mode"].eq(mode)]
    runs = (selected[["test_number", "test", "mode"]].drop_duplicates()
            .sort_values(["mode", "test_number"]))
    if runs.empty:
        print(f"No results found for mode={mode!r}")
        return
    ncols = 3
    nrows = int(np.ceil(len(runs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.2*nrows), squeeze=False)
    for ax, run in zip(axes.flat, runs.to_dict(orient="records")):
        group = selected[(selected["test"] == run["test"]) & (selected["mode"] == run["mode"])]
        matrix = confusion_table(group, normalize=normalize)
        sns.heatmap(matrix, annot=True, fmt=".1%" if normalize else "g", cmap="Blues",
                    vmin=0, vmax=1 if normalize else None, cbar=normalize, square=True, ax=ax)
        ax.set_title(f"{run['test']} | {run['mode']}")
        ax.set_xlabel("Predicted relation")
        ax.set_ylabel("True relation")
    for ax in axes.flat[len(runs):]:
        ax.axis("off")
    fig.suptitle("Row-normalized semantic confusion matrices" if normalize else "Semantic confusion matrices (counts)", y=1.01)
    plt.tight_layout()
    plt.show()

plot_confusion_matrices(results, CONFUSION_MODE, normalize=False)
plot_confusion_matrices(results, CONFUSION_MODE, normalize=True)

## 6. Analysis by camera viewpoint parsed from the image name

`cam_back`, `cam_front`, `cam_left`, and `cam_right` indicate where the camera is located relative to the person. This analysis is computed from the filename rather than from prompt text.

In [ ]:
camera_accuracy = (results[results["camera_view_from_path"].isin(CAMERA_ORDER)]
    .groupby(["test_number", "test", "mode", "camera_view_from_path"], observed=True)
    .agg(correct=("is_correct", "sum"), total=("is_correct", "size"), accuracy=("is_correct", "mean"))
    .reset_index().sort_values(["mode", "test_number", "camera_view_from_path"]))
camera_display = camera_accuracy.copy()
camera_display["accuracy"] = camera_display["accuracy"].map(lambda value: f"{value:.2%}")
print_table(camera_display)

for mode in camera_accuracy["mode"].unique():
    subset = camera_accuracy[camera_accuracy["mode"] == mode]
    pivot = subset.pivot(index="test", columns="camera_view_from_path", values="accuracy")
    pivot = pivot.reindex(columns=CAMERA_ORDER)
    order = subset[["test_number", "test"]].drop_duplicates().sort_values("test_number")["test"]
    pivot = pivot.reindex(order)
    plt.figure(figsize=(8, max(3, 0.55*len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap="YlGnBu", vmin=0, vmax=1)
    plt.title(f"Accuracy by camera viewpoint | {mode}")
    plt.xlabel("Camera location relative to person (parsed from filename)")
    plt.ylabel("Test")
    plt.tight_layout()
    plt.show()

## 7. Analysis by person-centric relation parsed from the image name

This is the direction of the object from the depicted person's perspective (`front`, `behind`, `left`, or `right`). It is independently parsed from the image path and checked against the CSV ground truth.

In [ ]:
person_accuracy = (results[results["person_relation_from_path"].isin(RELATION_ORDER)]
    .groupby(["test_number", "test", "mode", "person_relation_from_path"], observed=True)
    .agg(correct=("is_correct", "sum"), total=("is_correct", "size"), accuracy=("is_correct", "mean"))
    .reset_index().sort_values(["mode", "test_number", "person_relation_from_path"]))
person_display = person_accuracy.copy()
person_display["accuracy"] = person_display["accuracy"].map(lambda value: f"{value:.2%}")
print_table(person_display)

for mode in person_accuracy["mode"].unique():
    subset = person_accuracy[person_accuracy["mode"] == mode]
    pivot = subset.pivot(index="test", columns="person_relation_from_path", values="accuracy")
    pivot = pivot.reindex(columns=RELATION_ORDER)
    order = subset[["test_number", "test"]].drop_duplicates().sort_values("test_number")["test"]
    pivot = pivot.reindex(order)
    plt.figure(figsize=(8, max(3, 0.55*len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap="YlOrBr", vmin=0, vmax=1)
    plt.title(f"Accuracy by person-centric object relation | {mode}")
    plt.xlabel("Object relation from person's perspective (parsed from filename)")
    plt.ylabel("Test")
    plt.tight_layout()
    plt.show()

## 8. Joint camera-view × person-view analysis

These 4×4 heatmaps isolate viewpoint-transformation difficulty: rows describe where the camera is relative to the person, while columns describe where the object is from the person's perspective.

In [ ]:
JOINT_MODE = "non-thinking"  # change to "thinking" when desired
joint_data = results[results["mode"].eq(JOINT_MODE)]
runs = joint_data[["test_number", "test"]].drop_duplicates().sort_values("test_number")
ncols = 3
nrows = int(np.ceil(len(runs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.2*nrows), squeeze=False)
for ax, run in zip(axes.flat, runs.to_dict(orient="records")):
    group = joint_data[joint_data["test"].eq(run["test"])]
    pivot = group.pivot_table(index="camera_view_from_path", columns="person_relation_from_path",
                              values="is_correct", aggfunc="mean")
    pivot = pivot.reindex(index=CAMERA_ORDER, columns=RELATION_ORDER)
    sns.heatmap(pivot, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1,
                square=True, cbar=False, ax=ax)
    ax.set_title(run["test"])
    ax.set_xlabel("Object relation from person")
    ax.set_ylabel("Camera relative to person")
for ax in axes.flat[len(runs):]:
    ax.axis("off")
fig.suptitle(f"Joint viewpoint accuracy | {JOINT_MODE}", y=1.01)
plt.tight_layout()
plt.show()

## 9. Optional export

Exports tidy CSV tables that can be used in reports or downstream statistical analysis.

In [ ]:
if EXPORT_TABLES:
    output_dir = RESULT_ROOT / "analysis_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(output_dir / "qwen3_5_accuracy_summary.csv", index=False)
    quality.to_csv(output_dir / "qwen3_5_data_quality.csv", index=False)
    camera_accuracy.to_csv(output_dir / "qwen3_5_accuracy_by_camera_view.csv", index=False)
    person_accuracy.to_csv(output_dir / "qwen3_5_accuracy_by_person_relation.csv", index=False)
    print(f"Exported tables to {output_dir}")
else:
    print("EXPORT_TABLES=False; no files were written.")